In [25]:
import os
import sys
import fitz  # PyMuPDF
import re
import json

# Setup Path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
BE_DIR = os.path.join(PROJECT_ROOT, 'BE_generateBoBI')
UPLOAD_FOLDER = os.path.join(BE_DIR, "userinput")

# pdf_filename = "Biologi-BS-KLS-XI.pdf" 
pdf_filename = "Matematika_BS_KLS_XII_Rev.pdf" 
# pdf_filename = "Kelas X Bahasa Indonesia BS press.pdf" 
pdf_path = os.path.join(UPLOAD_FOLDER, pdf_filename)

print(f"Ready to extract index from: {pdf_path}")

Ready to extract index from: e:\file\skripsi\code\rake\BE_generateBoBI\userinput\Matematika_BS_KLS_XII_Rev.pdf


In [26]:
# --- BACA HALAMAN INDEKS DARI PDF ---
doc = fitz.open(pdf_path)
start_page = 199
end_page= 201
# end_page= len(doc)
index_text = ""
is_index = False

print(f"Mencari bagian Indeks di 15 halaman terakhir...")
for i in range(start_page, end_page):
    text = doc[i].get_text()
    if 'Indeks' in text or 'indeks' in text.lower():
        is_index = True
    if is_index:
        index_text += text + "\n"
        
print("Teks Indeks berhasil diekstrak! Panjang teks:", len(index_text), "karakter.")

Mencari bagian Indeks di 15 halaman terakhir...
Teks Indeks berhasil diekstrak! Panjang teks: 2237 karakter.


In [27]:
# --- PARSING TEKS INDEKS ---
# Kita akan membuat mapping: { "keyword": [nomor_halaman_1, nomor_halaman_2, ...] }
keyword_to_pages = {}

current_keyword = None

for line in index_text.split('\n'):
    line = line.strip()
    if not line: continue
    # Abaikan header 'Indeks' atau huruf abjad tunggal (A, B, C, dst)
    if line == 'Indeks' or (len(line) == 1 and line.isalpha()):
        continue
    
    # Deteksi pola: "keyword   10, 15, 20"
    match = re.match(r'^([A-Za-z][A-Za-z\s\-]*[A-Za-z])\s+(.*)$', line)
    if match:
        current_keyword = match.group(1).strip()
        numbers_part = match.group(2)
    else:
        # Menangani angka halaman yang turun ke baris baru (bisa mengandung tanda hubung range)
        if current_keyword and re.match(r'^[\d\,\s\-\–\—]+$', line):
            numbers_part = line
        elif re.match(r'^[A-Za-z][A-Za-z\s\-]+$', line):
            # Keyword panjang tanpa angka di baris ini
            current_keyword = line.strip()
            numbers_part = ""
        else:
            continue
    
    if current_keyword:
        kw_lower = current_keyword.lower()
        if kw_lower not in keyword_to_pages:
            keyword_to_pages[kw_lower] = []
            
        # Ekstrak angka tunggal maupun rentang halaman (e.g. 79-81)
        cleaned_numbers = numbers_part.replace('–', '-').replace('—', '-')
        parts = re.split(r'[\,\s]+', cleaned_numbers)
        for part in parts:
            part = part.strip()
            if not part:
                continue
            range_match = re.match(r'^(\d+)-(\d+)$', part)
            if range_match:
                start = int(range_match.group(1))
                end = int(range_match.group(2))
                if start <= end:
                    for p in range(start, end + 1):
                        if p not in keyword_to_pages[kw_lower]:
                            keyword_to_pages[kw_lower].append(p)
            else:
                nums = re.findall(r'\d+', part)
                for n in nums:
                    val = int(n)
                    if val not in keyword_to_pages[kw_lower]:
                        keyword_to_pages[kw_lower].append(val)

print(f"Berhasil memetakan indeks ke {len(keyword_to_pages)} kata kunci unik.")

Berhasil memetakan indeks ke 37 kata kunci unik.


In [28]:
# --- SIMPAN KE FILE JSON ---
# Extract bookname dynamically from pdf_filename
bookname = os.path.splitext(pdf_filename)[0].lower().replace(" ", "_").replace("-", "_")

# Create 'index' folder if it doesn't exist
index_dir = "index"
os.makedirs(index_dir, exist_ok=True)

output_json_path = os.path.join(index_dir, f"real_index_{bookname}.json")

# Sort the keys alphabetically
sorted_keyword_to_pages = dict(sorted(keyword_to_pages.items()))

# Sort the page lists
for kw in sorted_keyword_to_pages:
    sorted_keyword_to_pages[kw].sort()

with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(sorted_keyword_to_pages, f, indent=4, ensure_ascii=False)

print(f"Data Indeks Asli berhasil disimpan ke: {output_json_path}")
print(f"Silakan buka file '{output_json_path}' untuk mengecek atau mengedit kata kunci secara manual sebelum menjalankan pipeline eksperimen.")


Data Indeks Asli berhasil disimpan ke: index\real_index_matematika_bs_kls_xii_rev.json
Silakan buka file 'index\real_index_matematika_bs_kls_xii_rev.json' untuk mengecek atau mengedit kata kunci secara manual sebelum menjalankan pipeline eksperimen.
